# Conclusion

This chapter demonstrated that Convolutional Neural Networks (CNNs) can serve as accurate and computationally efficient surrogates for Finite Element Analysis in predicting magnetic field distributions. Through systematic investigation of architecture design, training strategies, and validation methodologies, this work establishes a comprehensive framework for deep learning-accelerated electromagnetic design.

**Chapter scope**: From fundamentals (Sections 2-4a) through CNN architecture (4b), training and results (4c), uncertainty quantification (5), and physics-informed approaches (6), this chapter provides a complete treatment of machine learning for electromagnetic field prediction.

## Key Achievements and Contributions

Every number in this conclusion is one the notebooks of this chapter printed or logged.

### 1. Accurate Surrogate Modeling for Electromagnetic Fields

The U-Nets of Section 4c predict the coil's magnetic field distribution from 1,000 FEMM solutions (700 for training) with:

- **Accuracy** on the 99 held-out field maps: 2.48% NMSE (baseline U-Net), 0.94% (dense U-Net), **0.17% (dilated U-Net)**
- **Speed**: 16 ms per 256×256 map on a laptop GPU (MX450), 159 ms on two CPU threads
- Compare: the FEMM solve that produced each map took 2.1 s on the same laptop
- **Speedup**: about 130× on the GPU, 13× on the CPU
- **Batch evaluation**: 1,000 coil designs in 16 s against 34 min with FEMM

The multilayer perceptron of Section 4a predicts C-core force with R² = 0.990 and RMSE = 1.66 N on 50 test designs (forces 0–81 N), from 500 FEMM solutions.

### 2. Physics-Motivated Architecture Design

**Critical choice**: Dilated convolutions {cite}`yu2015multi` for capturing long-range electromagnetic interactions:

**Physical motivation**:
- Biot-Savart law {cite}`sadiku2014elements`: Magnetic field at a point depends on distant current sources
- Dilated encoder blocks widen the receptive field at no parameter cost

**Empirical validation** (Section 4c):
- 5.4× error reduction compared to the identical network without dilation (0.94% → 0.17%)
- Validation NMSE below 1% at epoch 7 with dilation, epoch 44 without
- **Same parameter count** (3.35 M) — improved performance through architectural design, not capacity; the 7.76 M-parameter baseline was the worst of the three

**Implication**: Domain knowledge (physics) should guide neural network architecture, not just hyperparameter tuning.

### 3. Uncertainty Quantification: Demonstrated, Calibration Measured

Section 5 implements Monte Carlo dropout {cite}`gal2016dropout` and a deep ensemble on a synthetic regression task:

- **Uncertainty estimates**: per-prediction intervals from 50 stochastic forward passes
- **Calibration measured**: the nominal 90% interval covered **52%** of the test points — the intervals are too narrow out of the box
- **Decision framework**: criteria for when to trust a surrogate vs. revert to FEA, which must be preceded by a calibration check like the one above

**Importance for engineering**: Unlike image classification, incorrect field predictions can lead to motor failures. An uncertainty estimate is only useful once its coverage has been measured.

### 4. Physics-Informed Networks from 100 Observations

Sections 6a and 6b train on 100 sparse field observations of one coil plus the magnetostatic Poisson equation:

- Data-only network: 15.8% NMSE on the full FEMM field
- With nondimensionalised physics loss, hard boundary condition, radial input features and near-wire collocation: **0.14%**
- The decisive step, measured by ablation, was giving the network $r$ and $\ln r$ as inputs (5.9% → 0.24%); random Fourier features overfit the 100 points (55–159%)
- Reference: the analytic infinite-space wire field scores 0.9% because it ignores the walls

### 5. Design Guidelines

What the runs in this chapter support {cite}`goodfellow2016deep,bishop2006pattern`:

**Architecture**:
- Encoder-decoder structure {cite}`ronneberger2015unet` for dense spatial prediction
- Dilated convolutions in the encoder (5.4× accuracy improvement at equal size)
- Skip connections, following the U-Net literature (not ablated here)

**Training**:
- Adam optimizer {cite}`kingma2014adam` with cosine annealing {cite}`loshchilov2017sgdr`, learning rate 10⁻³, batch size 6, 50 epochs, best-validation checkpoint
- No dropout or weight decay was needed at 700 training maps and 50 epochs
- Batch normalization {cite}`ioffe2015batch` in the baseline U-Net only

**Data**:
- Latin Hypercube Sampling {cite}`mckay2000comparison` for space-filling designs
- 1,000 coil field solutions (70% / 20% / 10% split), 500 C-core force solutions, 100 PINN samples, all from FEMM {cite}`meeker2015femm` on a laptop in under an hour each

**Model capacity**:
- 3.35 M parameters were enough for the coil; the 7.76 M-parameter baseline did worse


## Future Research Directions

While this chapter demonstrates CNN and PINN surrogates on the coil problem, several research directions remain unexplored here. None of the numbers below are results; where a direction has not been tried in this chapter it says so.

### 1. Graph Neural Networks for Unstructured Meshes

**Motivation**: FEA uses unstructured meshes adapted to geometry complexity. CNNs require structured grids, so Section 4c rasterises every FEMM solution to a 256×256 grid.

**Graph Convolutional Networks (GCNs)** {cite}`goodfellow2016deep,bronstein2017geometric` offer potential advantages:
- Direct operation on FEA mesh topology (no interpolation required)
- Adaptive resolution at geometric features (fine mesh at material boundaries)
- Natural handling of complex geometries (arbitrary topologies)

**Challenges**:
- GCN training less mature than CNN (fewer established best practices)
- Computational efficiency unclear for large meshes
- Not attempted in this chapter

**Hybrid approach**: GCNs for geometry encoding, CNNs for field reconstruction.

### 2. Reduced Data Requirements

**Current cost**: 1,000 FEMM solutions took 34 min for the coil. The IPM motor of Section 3 takes 5.9 s per design, so a 1,000-design motor dataset would take about 1.6 h; the transformer and motor were not trained in this chapter.

**Semi-Supervised Learning** {cite}`goodfellow2016deep,chapelle2006semi`:
- Leverage unlabeled geometries (no FEA simulation needed)
- Self-supervised pre-training on geometric patterns, fine-tuning on a small labeled set
- Not tried here

**Active Learning** {cite}`settles2009active`:
- Select which geometries to simulate from the surrogate's uncertainty
- Iterative: train → identify uncertain regions → simulate those → retrain
- Section 5 shows the prerequisite, a calibrated uncertainty, is not automatic (52% coverage of a 90% interval)

**Transfer Learning** {cite}`goodfellow2016deep,weiss2016survey`:
- Pre-train on simpler problems (coil), fine-tune on complex problems (motor)
- Not tried here; Chapter 3 applies transfer learning to performance maps

### 3. Physics-Informed Neural Networks (PINNs)

**Current approach of Section 4**: Purely data-driven (learns from FEA examples)

**PINN approach** {cite}`raissi2019physics,barmada2020deep,cai2021physics`: Incorporate Maxwell's equations into the loss function

**Loss function augmentation**:
$$
L_{\text{total}} = L_{\text{data}} + \lambda_1 L_{\text{PDE}} + \lambda_2 L_{\text{BC}}
$$

where:
- $L_{\text{data}}$ = MSE between prediction and FEA (same as present work)
- $L_{\text{PDE}}$ = Violation of Maxwell's equations {cite}`sadiku2014elements`:
- $\nabla \times \mathbf{H} = \mathbf{J}$ (Ampere's law)
- $\nabla \cdot \mathbf{B} = 0$ (Gauss's law for magnetism)
- $L_{\text{BC}}$ = Boundary condition violations (material interfaces, domain boundaries)

**What Section 6 showed**: on one coil with 100 observations, 15.8% → 0.14% NMSE; the physics weight stopped mattering once the network received $r$ and $\ln r$ as inputs (0.24% at $\lambda_1 = 0.02$, 0.28% at $\lambda_1 = 1$).

**Open**:
- Nonlinear material properties (B-H saturation curves) enter the residual through $\mu(B)$; the transformer and motor were not attempted
- Multi-sample training conditioned on $(I, r_w)$ is supported by the code but was not run
- Second derivatives via automatic differentiation make each epoch costlier than a data-only step (3000 epochs took 110 s on the laptop GPU for 1,600 points)

### 4. Multi-Fidelity and Hybrid Modeling

**Motivation**: Different design phases require different accuracy-cost trade-offs

**Multi-fidelity framework** {cite}`peherstorfer2018survey,fernandez2016review`:
1. **Conceptual design**: Fast analytical models (reluctance networks, equivalent circuits)
2. **Preliminary design**: CNN surrogate (16 ms per map here, 0.17% NMSE on the coil)
3. **Detailed design**: 2D FEA (2–6 s per design for the problems in this chapter)
4. **Final validation**: 3D FEA with coupled thermal/structural analysis (not used in this chapter)

**Hybrid approaches**:
- **CNN as initializer**: Use the CNN prediction as the initial guess for the FEA solver (not tried here)
- **Adaptive fidelity**: Switch between models based on design phase and uncertainty
- High-confidence CNN predictions → skip FEA; low confidence → revert to FEA
- Requires the calibrated uncertainty that Section 5 shows must be checked first

### 5. Extension to Multi-Physics Coupling

**Current scope**: Magnetic field prediction only (electromagnetic)

**Future extensions**:
- **Thermal coupling**: Joule heating, core losses → temperature distribution → material properties
- **Structural coupling**: Maxwell stress → deformations → airgap changes
- **Acoustic coupling**: Force harmonics → vibrations → noise
- **Fluid coupling**: Cooling flow → heat transfer → temperature distribution

**Challenge**: Each physics domain requires:
- Additional, costlier coupled FEA simulations for training
- Multi-output CNN architectures (predict B-field, temperature, stress simultaneously)
- Coupled loss functions balancing accuracy across physics domains

**Potential**: A single CNN predicting all coupled fields could enable multi-objective optimisation (torque + efficiency + noise + cooling) that is impractical with coupled FEA alone.


## Practical Impact and Industrial Applications

The surrogate modelling approach of this chapter changes what a design study costs. The figures below are the ones measured on the laptop that ran the notebooks; the applications are what those figures make possible.

### Design Space Exploration and Optimization

**Enabled by a 16 ms prediction against a 2.1 s solve** (130× on the laptop GPU):

**Interactive Design Tools**:
- Field visualization updates in real time as geometry parameters are adjusted
- Engineers explore the design space without waiting for a solver run
- Immediate feedback on electromagnetic performance trade-offs

**Design Space Sweeps**:
- 1,000 coil designs: 16 s with the dilated U-Net, 34 min with FEMM
- Identify Pareto fronts in multi-objective optimization {cite}`deb2002fast`
- Discover non-intuitive optimal designs missed by one-at-a-time studies

**Gradient-Based Optimization**:
- The CNN is differentiable w.r.t. its inputs → gradients by automatic differentiation
- Enables gradient-based optimization (Adam, L-BFGS) that needs many evaluations

**Stochastic Optimization Algorithms** {cite}`goodfellow2016deep`:
- Genetic algorithms, particle swarm and Bayesian optimization all need thousands of evaluations; at 16 ms each, a run that needs 10,000 evaluations costs under 3 min of GPU time
- Chapters 4 and 5 use exactly such population-based and reinforcement-learning searches, with FEMM in the loop

### Industrial Relevance for Electric Machine Manufacturers

**Reduced Time-to-Market** {cite}`bilgin2019modeling`:
- Shorter design iterations when field evaluations stop being the bottleneck
- Rapid prototyping: evaluate radical design concepts before committing solver time

**Cost Savings**:
- The surrogate runs on a laptop GPU; the FEMM dataset and the training each took about half an hour on the same machine
- Engineering time: less waiting for simulation results

**Innovation Enablement**:
- Explore larger design spaces than solver time allows
- Consider novel topologies and multi-physics trade-offs

**Quality Improvement**:
- Better optimized designs (explored larger space)
- Reduced margin where predictions are accurate, provided the uncertainty is calibrated (Section 5)

**Specific applications**:
- Electric vehicle traction motors: Range optimization via efficiency improvements
- Industrial motors: IE4/IE5 efficiency compliance (Section 1)
- Aerospace actuators: Power density maximization under weight constraints
- Renewable energy generators: Cost reduction through design optimization

### Research and Educational Value

**Accelerated Research**:
- Parametric studies across wide parameter ranges
- Sensitivity analysis identifying critical design parameters

**Educational Tools**:
- Every notebook in this chapter runs on a laptop: the coil dataset, the U-Nets and the PINN together take under three hours
- Students explore designs without waiting for FEA
- Immediate feedback on parameter effects

**Benchmarking Platform**:
- The datasets, trained weights and executed notebooks of this chapter are reproducible artefacts
- Comparison baseline for future methods (GNNs, PINN variants)

### Cost on This Book's Hardware

- **Data**: 1,000 FEMM coil solutions, 34 min; 500 C-core force solutions, 32 min; 100 PINN samples
- **Training**: 36–37 min per `UNetField` on the MX450; 13 min for the baseline U-Net on a free Kaggle T4; 110 s for the PINN
- **Inference**: 16 ms per field map; a 1,000-design sweep in 16 s
- **Break-even**: data plus training (71 min) equal about 2,000 FEMM solves; every evaluation after that saves 2 s


## Limitations and Path Forward

While this chapter demonstrates CNN and PINN surrogates for electromagnetic fields, important limitations and research opportunities remain.

### Current Limitations

**1. One Problem Trained**:
- **Issue**: The U-Nets and the PINN were trained on the linear air-core coil only; the transformer and IPM motor of Section 3 were simulated but not learned
- **Impact**: Nothing in this chapter measures surrogate accuracy under magnetic saturation
- **Path**: The data scripts take `--problem-type transformer` and `ipm_motor`; the motor costs 5.9 s per FEMM design on the laptop

**2. Structured Grid Constraint**:
- **Issue**: CNNs require structured grids; FEA uses unstructured meshes
- **Impact**: Interpolation to 256×256 adds a step and can lose detail at sharp boundaries
- **Future**: Graph Neural Networks may natively handle unstructured meshes {cite}`bronstein2017geometric`

**3. Fixed Resolution**:
- **Issue**: Same grid resolution (256×256) for all predictions
- **Impact**: Overkill for simple geometries, insufficient for complex geometries
- **Future**: Multiple resolutions or adaptive refinement integrated with CNN predictions

**4. Interpolation vs. Extrapolation**:
- **Issue**: CNN predicts accurately within training distribution, degrades outside
- **Impact**: Novel geometries far from training data have unreliable predictions
- **Mitigation**: Uncertainty quantification identifies unreliable predictions (Section 5) — once calibrated; the dropout interval covered 52% instead of 90% on the synthetic task
- **Future**: Physics-informed approaches may improve extrapolation {cite}`raissi2019physics`

**5. Uncalibrated Uncertainty**:
- **Issue**: Monte Carlo dropout intervals were too narrow on the synthetic task of Section 5
- **Impact**: An automated trust-the-surrogate decision cannot rest on the raw interval
- **Future**: Calibration on held-out data, ensembles, or conformal methods

**6. 2D Limitation**:
- **Current scope**: 2D planar geometries only
- **Impact**: Cannot model true 3D effects (end windings, skew, 3D flux paths)
- **Computational challenge**: a 256×256×256 field is 16.8 million values per sample
- **Future**: 3D CNNs or sparse representations for memory efficiency

**7. Single Physics Domain**:
- **Current scope**: Electromagnetic fields only
- **Missing**: Thermal (temperature), structural (stress), acoustic (noise)
- **Future**: Multi-output networks predicting all coupled fields simultaneously

### Hybrid Approaches

**CNN as FEA Initializer**:
- Use the CNN prediction as the initial guess for the FEA solver
- **Implementation**: Interpolate CNN grid → FEA mesh → solve
- **Status**: not tried in this chapter

**Adaptive Fidelity Selection**:
- Switch between models based on uncertainty and design phase
- **Logic**:
- Low uncertainty + preliminary design → use CNN (fast)
- High uncertainty → use FEA (accurate)
- Critical design phase → always use FEA (conservative)
- **Implementation**: Uncertainty threshold tuned to risk tolerance, after calibration

**Multi-Fidelity Optimization**:
- Combine analytical, CNN, FEA models in optimization loop {cite}`peherstorfer2018survey`
- **Strategy**: Use cheap models (CNN) for broad exploration, expensive models (FEA) for refinement

### Research Recommendations

Based on this chapter's findings, future research should prioritize:

**1. High Priority**:
- **Train the nonlinear problems**: the transformer and motor datasets are one script call away and would show whether dilation and the 3.35 M-parameter U-Net hold up under saturation
- **Calibrate the uncertainty**: measure coverage on held-out FEMM data, not only on synthetic tasks

**2. Medium Priority**:
- **Transfer learning**: Pre-train on the coil, fine-tune on the motor (not tried here; Chapter 3 does this for performance maps)
- **Physics-informed approaches** at scale: the Section 6 result (0.14% from 100 observations) is for one linear coil; nonlinear materials and multi-sample conditioning are open
- **Graph neural networks**: Handle unstructured meshes natively

**3. Long-Term Goals**:
- **3D field prediction**: Extend to full 3D electromagnetic problems
- **Multi-physics coupling**: Predict electromagnetic, thermal, structural fields simultaneously

### Path Forward: Integration into Design Workflows

**Recommended deployment strategy**:

**Phase 1: Pilot studies**
- Train the surrogate on an existing FEA database
- Validate on historical design projects; measure accuracy and interval coverage

**Phase 2: Parallel operation**
- Run the surrogate alongside FEA for new projects
- Compare predictions, refine models, build engineering trust

**Phase 3: Primary tool**
- Surrogate becomes the primary prediction tool
- FEA reserved for validation and edge cases

**Success criteria** (proposed, not measured in this chapter):
- Predictions within the engineering tolerance of the application on held-out designs
- Interval coverage that matches its nominal level
- Design throughput measured in projects per year


## Final Remarks and Broader Impact

This chapter demonstrated that deep learning, specifically Convolutional Neural Networks, can serve as accurate and computationally efficient surrogates for electromagnetic finite element analysis on the problem it was tested on. The work establishes both the theoretical foundations and practical methodologies for CNN-based field prediction, and every result is reproduced by the notebooks.

### Core Achievements

**[OK] Accurate prediction**: 0.17% normalized mean squared error on 99 held-out coil field maps (dilated U-Net); 0.94% dense, 2.48% baseline

**[OK] Computational efficiency**: 16 ms per field map on a laptop GPU against 2.1 s per FEMM solve — about 130×

**[OK] Physics-motivated design**: Dilated convolutions capture long-range electromagnetic interactions (5.4× accuracy improvement at equal parameter count)

**[OK] Uncertainty quantification**: Monte Carlo dropout {cite}`gal2016dropout` and ensembles implemented, and their calibration measured (52% coverage of a nominal 90% interval)

**[OK] Physics-informed learning**: 0.14% NMSE from 100 observations and the Poisson equation, against 15.8% from the data alone

**[OK] Design guidelines**: Best practices for architecture, training, and validation, each tied to a run in this chapter

### Key Insight: Domain Knowledge Enables Better Architectures

**Most important finding**: Physics understanding (Biot-Savart law → dilated convolutions) provided a 5.4× accuracy improvement — far exceeding the gain from the 2.3× larger baseline network, which was the worst of the three. In the PINN the same lesson appears as input features: giving the network $r$ and $\ln r$ did more than any physics weight.

**Implication for ML in science/engineering**: Domain expertise should guide architectural choices, not just data collection and hyperparameter optimization. The most successful ML applications in physics will come from deep collaboration between domain experts and ML researchers {cite}`goodfellow2016deep,carleo2019machine`.

### Reliability Through Uncertainty Quantification

**Critical for engineering deployment**: Unlike image classification where mistakes are inconsequential, incorrect electromagnetic predictions can lead to motor failures, safety issues, or costly redesigns.

**Solution presented**: Bayesian deep learning via Monte Carlo dropout {cite}`gal2016dropout` provides per-prediction uncertainty estimates, enabling:
- Automated decision: When to trust CNN (low uncertainty) vs. revert to FEA (high uncertainty)
- Risk management: Conservative predictions for critical designs
- Active learning: Identify geometries needing FEA validation

**Caveat measured here**: the raw dropout interval was too narrow (52% coverage at a nominal 90%); calibration is a required step before any of the above.

### Advancing Electromagnetic Device Design

The broader impact extends beyond computational speedup:

**1. Democratization of Design Tools**:
- Everything in this chapter ran on a laptop and a free cloud GPU
- Interactive tools make electromagnetic design more accessible
- Educational value for students learning field theory

**2. Enabling New Design Methodologies**:
- Large-scale design space exploration (1,000 designs in 16 s)
- Multi-objective optimization with thousands of evaluations
- Rapid prototyping of radical design concepts

**3. Accelerating Clean Energy Transition**:
- Faster development of efficient electric motors (IE4/IE5 compliance, Section 1)
- Improved EV traction motors (range extension via efficiency gains)
- Cost reduction through design optimization (renewable energy generators)

**4. Scientific Discovery**:
- CNN learned representations reveal design insights
- Automated feature extraction identifies important geometric parameters

### Closing Perspective

This work demonstrates that deep learning is not merely a "black box" tool but, when:
- **Properly designed** with domain knowledge (dilated convolutions for long-range interactions; radial inputs for the PINN)
- **Validated on held-out data** (three networks on one problem, every number produced by the notebooks)
- **Equipped with uncertainty quantification** whose calibration has been checked
- **Grounded in physics** (architectural choices motivated by Maxwell's equations)

...can serve as a **reliable, physics-aware surrogate** for electromagnetic simulations {cite}`goodfellow2016deep,bishop2006pattern,carleo2019machine`.

The combination of **computational efficiency** (130× faster on the laptop), **accuracy** (0.17% error on the coil), **measured uncertainty** and **physics grounding** (dilated convolutions for Biot-Savart; the Poisson residual in the PINN) opens new possibilities for electromagnetic device design — from rapid prototyping to multi-objective optimization — toward more efficient and higher-performing electric machines crucial for global electrification and decarbonization {cite}`bilgin2019modeling,IEA_Global_EV_Outlook_2024`.

---

**Summary of contributions**:
1. U-Net with dilated encoder convolutions for electromagnetic field prediction: 0.17% NMSE on the coil
2. Controlled comparison of three networks under identical conditions: dilation 5.4×, capacity no help
3. Physics-motivated design principles validated empirically
4. Uncertainty quantification with its calibration measured
5. Physics-informed network reaching 0.14% from 100 observations, with an ablation of every ingredient
6. Design guidelines for surrogate modeling in computational electromagnetics
7. A reproducible pipeline: FEMM data, training and evaluation on a laptop

This framework provides a foundation for future work integrating machine learning with computational electromagnetics, with extensions to the nonlinear transformer and motor problems, 3D geometries and multi-physics coupling.

## References

```{bibliography}
:filter: docname in docnames
:style: unsrt
```